In [2]:
import gdsfactory as gf
gf.gpdk.PDK.activate()

In [2]:
#from upvfab.sin300.cband import PDK, cells  # celdas --> componentes
                                            # PDK --> materiales, distancias seguras, layers

In [19]:
nm=1e-3
layer_wg = "WG"
layer_box = "FLOORPLAN"
dieW = 5000

width=500

border=250
L=49536.082*nm

xs = gf.cross_section.strip(width=width, layer = 'WG')
mmi2x2 = gf.components.mmi2x2(width_taper=1, length_taper=10, length_mmi=34, width_mmi=2.5, gap_mmi=0.25, cross_section=xs)
wvg_n=gf.components.straight(length=3, cross_section=xs)


/home/au/Topic-6-WDM-Cascaded-MZIs/.venv/lib/python3.12/site-packages/gdsfactory/components/waveguides/straight.py:33: UserWarning: {'width': 2.5} ignored for cross_section 'xs_974aca1a'
  x = gf.get_cross_section(cross_section, width=width)


In [26]:
c = gf.components.spiral_racetrack_fixed_length(length=L, in_out_port_spacing=50, n_straight_sections=2, min_spacing=5, bend='bend_circular', bend_s='bend_s', cross_section=xs)
c.draw_ports()
c.plot()

# Manejamos la minima longitud del espiral moviendo el n_straight_sections --> por si necesitamos longitudes menores

ValueError: A value (49.536082) in x_new is below the interpolation range's minimum value (595.3778506397813).

In [11]:
#Spiral Volteada


gf.clear_cache()
@gf.cell  # importante ponerle esto, para decirle que es una celfa de gdsfactory
def spiral(L=400,n_straight_spiral=4,in_out_dist=100)-> gf.Component:
    
    #componentes
    array_ring = gf.Component()
    spiral=gf.components.spiral_racetrack_fixed_length(length=L, in_out_port_spacing=in_out_dist, n_straight_sections=n_straight_spiral, min_spacing=5, bend='bend_circular', bend_s='bend_s', cross_section=xs)
    mmi2x2 = gf.components.mmi2x2(width_taper=1, length_taper=10, length_mmi=5.5, width_mmi=2.5, gap_mmi=0.25, cross_section=xs)

    _mmi1=array_ring.add_ref(mmi2x2)
    _spiral=array_ring.add_ref(spiral)
    _mmi2=array_ring.add_ref(mmi2x2)
    

    _spiral.connect('o1', _mmi1['o4'])
    _mmi2.connect('o1', _spiral['o2'])

    gf.routing.route_single_sbend(array_ring,port1=_mmi1['o4'], port2=_spiral['o1'], cross_section=xs)
    gf.routing.route_single_sbend(array_ring,port1=_spiral['o2'], port2=_mmi2['o1'], cross_section=xs)
    gf.routing.route_single_sbend(array_ring,port1=_mmi1['o3'], port2=_mmi2['o2'], cross_section=xs)




    # Adding ports to a component 
    
    array_ring.add_port(name="o1", center=[_mmi1['o1'].dx,_mmi1['o1'].dy], width=width, orientation=180, layer=layer_wg)
    array_ring.add_port(name="o2", center=[_mmi1['o2'].dx,_mmi1['o2'].dy], width=width, orientation=180, layer=layer_wg)
    array_ring.add_port(name="o3", center=[_mmi2['o3'].dx,_mmi2['o3'].dy], width=width, orientation=0, layer=layer_wg)
    array_ring.add_port(name="o4", center=[_mmi2['o4'].dx,_mmi2['o4'].dy], width=width, orientation=0, layer=layer_wg)
    
    #caminos.add_port(name="org", center=[die_ref["block@org"].dx ,die_ref["block@org"].dy], width=1, orientation=0, layer=layer_box)
    #caminos.draw_ports()

    return array_ring
    


In [12]:
#Spiral normal

@gf.cell  # importante ponerle esto, para decirle que es una celfa de gdsfactory
def spiral_s(L=400,n_straight_spiral=4,in_out_dist=100)-> gf.Component:
    
    #componentes
    array_ring = gf.Component()
    spiral=gf.components.spiral_racetrack_fixed_length(length=L, in_out_port_spacing=in_out_dist, n_straight_sections=n_straight_spiral, min_spacing=5, bend='bend_circular', bend_s='bend_s', cross_section=xs)
    mmi2x2 = gf.components.mmi2x2(width_taper=1, length_taper=10, length_mmi=5.5, width_mmi=2.5, gap_mmi=0.25, cross_section=xs)

    _mmi1=array_ring.add_ref(mmi2x2)
    _spiral=array_ring.add_ref(spiral)
    _mmi2=array_ring.add_ref(mmi2x2)
    

    _spiral.connect('o1', _mmi1['o3'])
    _mmi2.connect('o2', _spiral['o2'])

    gf.routing.route_single_sbend(array_ring,port1=_mmi1['o3'], port2=_spiral['o1'], cross_section=xs)
    gf.routing.route_single_sbend(array_ring,port1=_spiral['o2'], port2=_mmi2['o2'], cross_section=xs)
    gf.routing.route_single_sbend(array_ring,port1=_mmi1['o4'], port2=_mmi2['o1'], cross_section=xs)




    # Adding ports to a component 
    
    array_ring.add_port(name="o1", center=[_mmi1['o1'].dx,_mmi1['o1'].dy], width=width, orientation=180, layer=layer_wg)
    array_ring.add_port(name="o2", center=[_mmi1['o2'].dx,_mmi1['o2'].dy], width=width, orientation=180, layer=layer_wg)
    array_ring.add_port(name="o3", center=[_mmi2['o3'].dx,_mmi2['o3'].dy], width=width, orientation=0, layer=layer_wg)
    array_ring.add_port(name="o4", center=[_mmi2['o4'].dx,_mmi2['o4'].dy], width=width, orientation=0, layer=layer_wg)
    
    #caminos.add_port(name="org", center=[die_ref["block@org"].dx ,die_ref["block@org"].dy], width=1, orientation=0, layer=layer_box)
    #caminos.draw_ports()

    return array_ring
    


In [13]:
#circuit
L=4*200

@gf.cell  # importante ponerle esto, para decirle que es una celfa de gdsfactory
def circuit(mmiy=4, r_anilos_3=25, dis_min=1,movX=50)-> gf.Component:
    
    #componentes
    circuit = gf.Component()

    _spiral1=circuit.add_ref(spiral(L=L,n_straight_spiral=4,in_out_dist=150))
    _spiral2=circuit.add_ref(spiral_s(L=L/2,n_straight_spiral=2,in_out_dist=100))
    _spiral3=circuit.add_ref(spiral_s(L=L/2,n_straight_spiral=2,in_out_dist=100))

    _spiral4=circuit.add_ref(spiral_s(L=L/4,n_straight_spiral=2,in_out_dist=50))
    _spiral5=circuit.add_ref(spiral_s(L=L/4,n_straight_spiral=2,in_out_dist=50))
    
    _spiral6=circuit.add_ref(spiral_s(L=L/4,n_straight_spiral=2,in_out_dist=50))
    _spiral7=circuit.add_ref(spiral_s(L=L/4,n_straight_spiral=2,in_out_dist=50))

    _spiral1.rotate(180)
    #-------------------------------
    _spiral2.connect('o1', _spiral1['o1'])
    _spiral3.connect('o1', _spiral1['o2'])
    _spiral2.dmovey(2*(2*mmiy+r_anilos_3+dis_min)).dmovex(movX)
    _spiral3.dmovey(-2*(2*mmiy+r_anilos_3+dis_min)).dmovex(movX)

    #------------------------------
    _spiral4.connect('o1', _spiral2['o3'])
    _spiral5.connect('o1', _spiral2['o4'])
    _spiral4.dmovey(2*mmiy+r_anilos_3+dis_min).dmovex(movX)
    _spiral5.dmovey(-(2*mmiy+r_anilos_3+dis_min)).dmovex(movX)

    #------------------------------
    _spiral6.connect('o1', _spiral3['o3'])
    _spiral7.connect('o1', _spiral3['o4'])
    _spiral6.dmovey(2*mmiy+r_anilos_3+dis_min).dmovex(movX)
    _spiral7.dmovey(-(2*mmiy+r_anilos_3+dis_min)).dmovex(movX)
    


        
    gf.routing.route_single_sbend(circuit,port1=_spiral1['o1'], port2=_spiral2['o1'], cross_section=xs)
    gf.routing.route_single_sbend(circuit,port1=_spiral1['o2'], port2=_spiral3['o1'], cross_section=xs)
    
    gf.routing.route_single_sbend(circuit,port1=_spiral2['o3'], port2=_spiral4['o1'], cross_section=xs)
    gf.routing.route_single_sbend(circuit,port1=_spiral2['o4'], port2=_spiral5['o1'], cross_section=xs)
    
    gf.routing.route_single_sbend(circuit,port1=_spiral3['o3'], port2=_spiral6['o1'], cross_section=xs)
    gf.routing.route_single_sbend(circuit,port1=_spiral3['o4'], port2=_spiral7['o1'], cross_section=xs)




    # Adding ports to a component 
    circuit.add_port(name="in1", center=[_spiral1['o4'].dx,_spiral1['o4'].dy], width=width, orientation=180, layer=layer_wg)
    circuit.add_port(name="in2", center=[_spiral1['o3'].dx,_spiral1['o3'].dy], width=width, orientation=180, layer=layer_wg)
    
    circuit.add_port(name="o1", center=[_spiral4['o3'].dx,_spiral4['o3'].dy], width=width, orientation=0, layer=layer_wg)
    circuit.add_port(name="o2", center=[_spiral4['o4'].dx,_spiral4['o4'].dy], width=width, orientation=0, layer=layer_wg)
    
    circuit.add_port(name="o3", center=[_spiral5['o3'].dx,_spiral5['o3'].dy], width=width, orientation=0, layer=layer_wg)
    circuit.add_port(name="o4", center=[_spiral5['o4'].dx,_spiral5['o4'].dy], width=width, orientation=0, layer=layer_wg)
    
    circuit.add_port(name="o5", center=[_spiral6['o3'].dx,_spiral6['o3'].dy], width=width, orientation=0, layer=layer_wg)
    circuit.add_port(name="o6", center=[_spiral6['o4'].dx,_spiral6['o4'].dy], width=width, orientation=0, layer=layer_wg)
    
    circuit.add_port(name="o7", center=[_spiral7['o3'].dx,_spiral7['o3'].dy], width=width, orientation=0, layer=layer_wg)
    circuit.add_port(name="o8", center=[_spiral7['o4'].dx,_spiral7['o4'].dy], width=width, orientation=0, layer=layer_wg)
    
    circuit.draw_ports()

    return circuit
    


In [14]:
cam = gf.Component()
cx = cam.add_ref(circuit())

cam.show()
cam.plot()

ValueError: A value (800.0) in x_new is below the interpolation range's minimum value (839.7057299955624).